In [17]:
# download dataset from the UCI website
!curl -o uci-labelled-sentences.zip https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip

# unzip dataset in Colab
!unzip uci-labelled-sentences.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 84188    0 84188    0     0   179k      0 --:--:-- --:--:-- --:--:--  179k
Archive:  uci-labelled-sentences.zip
replace sentiment labelled sentences/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [18]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM
from keras.callbacks import EarlyStopping

In [19]:
df_list = []

# Yelp
df_yelp = pd.read_csv('sentiment labelled sentences/yelp_labelled.txt', names=['sentence', 'label'], sep='\t')
df_yelp['source'] = 'yelp'
df_list.append(df_yelp)

# Amazon
df_amazon = pd.read_csv('sentiment labelled sentences/amazon_cells_labelled.txt', names=['sentence', 'label'], sep='\t')
df_amazon['source'] = 'amazon'
df_list.append(df_amazon)

# IMDB
df_imdb = pd.read_csv('sentiment labelled sentences/imdb_labelled.txt', names=['sentence', 'label'], sep='\t')
df_imdb['source'] = 'imdb'
df_list.append(df_imdb)

# Concatenate the dataframes
df = pd.concat(df_list)

df.head()

,sentence,label,source
0,Wow... Loved this place.,1,yelp
1,Crust is not good.,0,yelp
2,Not tasty and the texture was just nasty.,0,yelp
3,Stopped by during the late May bank holiday of...,1,yelp
4,The selection on the menu was great and so wer...,1,yelp


In [20]:
max_features = 2000
tokenizer = Tokenizer(num_words=max_features, split=' ')
tokenizer.fit_on_texts(df['sentence'].values)
X = tokenizer.texts_to_sequences(df['sentence'].values)
X = pad_sequences(X)
y = df['label'].values

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.12)

In [22]:
def create_model():
  model = Sequential()
  model.add(Embedding(max_features, 64, input_length=X.shape[1]))
  model.add(LSTM(16))
  model.add(Dense(1, activation='sigmoid'))
  model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
  return model

model = create_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [23]:
model.fit(X_train, y_train, epochs=6, batch_size=16, validation_data=(X_test, y_test), callbacks = [EarlyStopping(monitor='val_accuracy', min_delta=0.001, patience=2, verbose=1)])

Epoch 1/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 85s 537ms/step - accuracy: 0.6435 - loss: 0.6380 - val_accuracy: 0.7394 - val_loss: 0.5335
Epoch 2/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 77s 506ms/step - accuracy: 0.8507 - loss: 0.3696 - val_accuracy: 0.8152 - val_loss: 0.3937
Epoch 3/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 76s 502ms/step - accuracy: 0.8867 - loss: 0.2892 - val_accuracy: 0.8182 - val_loss: 0.4344
Epoch 4/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 77s 505ms/step - accuracy: 0.9396 - loss: 0.1802 - val_accuracy: 0.8364 - val_loss: 0.3841
Epoch 5/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 80s 495ms/step - accuracy: 0.9607 - loss: 0.1187 - val_accuracy: 0.8515 - val_loss: 0.4252
Epoch 6/6
152/152 ━━━━━━━━━━━━━━━━━━━━ 75s 495ms/step - accuracy: 0.9777 - loss: 0.0782 - val_accuracy: 0.8394 - val_loss: 0.4911


In [24]:
model.save("uci_sentimentanalysis.h5")

with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.DEFAULT_PROTOCOL)

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sentiment Analysis</title>
    <link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Mono:wght@300;400;500&display=swap" rel="stylesheet">
    <style>
        *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

        :root {
            --ink: #1a1714;
            --paper: #f5f0e8;
            --cream: #ede7d5;
            --accent: #c84b2f;
            --muted: #8a7f72;
            --pos: #2d6a4f;
            --neg: #c84b2f;
            --neu: #5c6b8a;
            --comp: #7b4f9e;
        }

        body {
            background-color: var(--paper);
            color: var(--ink);
            font-family: 'DM Mono', monospace;
            min-height: 100vh;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            padding: 2rem;
            position: relative;
            overflow-x: hidden;
        }

        body::before {
            content: '';
            position: fixed;
            inset: 0;
            background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 256 256' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='noise'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23noise)' opacity='0.04'/%3E%3C/svg%3E");
            pointer-events: none;
            z-index: 0;
            opacity: 0.5;
        }

        .container {
            width: 100%;
            max-width: 640px;
            position: relative;
            z-index: 1;
            animation: fadeUp 0.6s ease both;
        }

        @keyframes fadeUp {
            from { opacity: 0; transform: translateY(20px); }
            to   { opacity: 1; transform: translateY(0); }
        }

        .header {
            margin-bottom: 2.5rem;
            border-bottom: 1.5px solid var(--ink);
            padding-bottom: 1.25rem;
        }

        .header-label {
            font-family: 'DM Mono', monospace;
            font-size: 0.65rem;
            font-weight: 500;
            letter-spacing: 0.18em;
            text-transform: uppercase;
            color: var(--muted);
            margin-bottom: 0.5rem;
        }

        h1 {
            font-family: 'DM Serif Display', serif;
            font-size: clamp(2rem, 5vw, 3rem);
            line-height: 1.05;
            letter-spacing: -0.02em;
            color: var(--ink);
        }

        h1 em {
            color: var(--accent);
            font-style: italic;
        }

        .form-card {
            background: var(--cream);
            border: 1.5px solid var(--ink);
            border-radius: 2px;
            padding: 1.75rem;
            margin-bottom: 1.5rem;
            box-shadow: 4px 4px 0px var(--ink);
        }

        .field-label {
            display: block;
            font-size: 0.65rem;
            font-weight: 500;
            letter-spacing: 0.15em;
            text-transform: uppercase;
            color: var(--muted);
            margin-bottom: 0.6rem;
        }

        textarea {
            width: 100%;
            background: var(--paper);
            border: 1.5px solid var(--ink);
            border-radius: 2px;
            padding: 0.85rem 1rem;
            font-family: 'DM Mono', monospace;
            font-size: 0.85rem;
            color: var(--ink);
            resize: vertical;
            min-height: 130px;
            outline: none;
            transition: box-shadow 0.2s;
            line-height: 1.6;
        }

        textarea::placeholder { color: var(--muted); }

        textarea:focus {
            box-shadow: 3px 3px 0px var(--accent);
            border-color: var(--accent);
        }

        .submit-btn {
            display: inline-flex;
            align-items: center;
            gap: 0.5rem;
            margin-top: 1.1rem;
            background: var(--ink);
            color: var(--paper);
            border: 1.5px solid var(--ink);
            border-radius: 2px;
            padding: 0.65rem 1.4rem;
            font-family: 'DM Mono', monospace;
            font-size: 0.75rem;
            font-weight: 500;
            letter-spacing: 0.1em;
            text-transform: uppercase;
            cursor: pointer;
            transition: background 0.18s, color 0.18s, box-shadow 0.18s;
        }

        .submit-btn:hover {
            background: var(--accent);
            border-color: var(--accent);
            box-shadow: 3px 3px 0 var(--ink);
        }

        .results {
            animation: fadeUp 0.5s 0.1s ease both;
        }

        .results-label {
            font-size: 0.65rem;
            font-weight: 500;
            letter-spacing: 0.18em;
            text-transform: uppercase;
            color: var(--muted);
            margin-bottom: 0.85rem;
        }

        .scores-grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 0.85rem;
        }

        .score-card {
            background: var(--cream);
            border: 1.5px solid var(--ink);
            border-radius: 2px;
            padding: 1.1rem 1.2rem;
            box-shadow: 3px 3px 0 var(--ink);
            position: relative;
            overflow: hidden;
        }

        .score-card::before {
            content: '';
            position: absolute;
            top: 0; left: 0;
            width: 3px;
            height: 100%;
        }

        .score-card.pos::before { background: var(--pos); }
        .score-card.neg::before { background: var(--neg); }
        .score-card.neu::before { background: var(--neu); }
        .score-card.comp::before { background: var(--comp); }
        .score-card.custom::before { background: var(--accent); }
        .score-card.custom { grid-column: 1 / -1; }

        .score-name {
            font-size: 0.6rem;
            font-weight: 500;
            letter-spacing: 0.15em;
            text-transform: uppercase;
            color: var(--muted);
            margin-bottom: 0.35rem;
        }

        .score-value {
            font-family: 'DM Serif Display', serif;
            font-size: 1.9rem;
            line-height: 1;
            color: var(--ink);
        }

        .score-bar {
            margin-top: 0.6rem;
            height: 3px;
            background: rgba(26,23,20,0.1);
            border-radius: 2px;
            overflow: hidden;
        }

        .score-bar-fill {
            height: 100%;
            border-radius: 2px;
            transition: width 0.8s cubic-bezier(0.16, 1, 0.3, 1);
        }

        .pos .score-bar-fill  { background: var(--pos); }
        .neg .score-bar-fill  { background: var(--neg); }
        .neu .score-bar-fill  { background: var(--neu); }
        .comp .score-bar-fill { background: var(--comp); }
        .custom .score-bar-fill { background: var(--accent); }

        .footer {
            margin-top: 2rem;
            font-size: 0.62rem;
            letter-spacing: 0.1em;
            color: var(--muted);
            text-align: center;
            text-transform: uppercase;
        }
    </style>
</head>
<body>
    <div class="container">

        <div class="header">
            <h1>Sentiment<br><em>Analysis</em></h1>
        </div>

        <form method="POST">
            <div class="form-card">
                <label class="field-label" for="user_text">Enter text to analyze</label>
                <textarea
                    id="user_text"
                    name="user_text"
                    rows="5"
                    placeholder="Type or paste any text here…"
                ></textarea>
                <button type="submit" class="submit-btn">
                    ↳ Analyze
                </button>
            </div>
        </form>

        {% if sentiment %}
        <div class="results">
            <p class="results-label">Analysis Results</p>
            <div class="scores-grid">

                <div class="score-card pos">
                    <p class="score-name">Positive</p>
                    <p class="score-value">{{ "%.1f"|format(sentiment['pos'] * 100) }}%</p>
                    <div class="score-bar">
                        <div class="score-bar-fill" style="width: {{ sentiment['pos'] * 100 }}%"></div>
                    </div>
                </div>

                <div class="score-card neg">
                    <p class="score-name">Negative</p>
                    <p class="score-value">{{ "%.1f"|format(sentiment['neg'] * 100) }}%</p>
                    <div class="score-bar">
                        <div class="score-bar-fill" style="width: {{ sentiment['neg'] * 100 }}%"></div>
                    </div>
                </div>

                <div class="score-card neu">
                    <p class="score-name">Neutral</p>
                    <p class="score-value">{{ "%.1f"|format(sentiment['neu'] * 100) }}%</p>
                    <div class="score-bar">
                        <div class="score-bar-fill" style="width: {{ sentiment['neu'] * 100 }}%"></div>
                    </div>
                </div>

                <div class="score-card comp">
                    <p class="score-name">Compound</p>
                    <p class="score-value">{{ "%.2f"|format(sentiment['compound']) }}</p>
                    <div class="score-bar">
                        <div class="score-bar-fill" style="width: {{ ((sentiment['compound'] + 1) / 2) * 100 }}%"></div>
                    </div>
                </div>

                <div class="score-card custom">
                    <p class="score-name">Custom Model — Positive Probability</p>
                    <p class="score-value">{{ "%.0f"|format(sentiment['custom model positive'] * 100) }}%</p>
                    <div class="score-bar">
                        <div class="score-bar-fill" style="width: {{ sentiment['custom model positive'] * 100 }}%"></div>
                    </div>
                </div>

            </div>
        </div>
        {% endif %}

        <p class="footer">VADER + Custom Keras Model &nbsp;·&nbsp; Sentiment Analysis</p>
    </div>
</body>
</html>



In [10]:
import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import sequence

In [11]:
model = load_model('uci_sentimentanalysis.h5')
with open('tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)

In [12]:
negative_test_input = """
this is bad junk.
"""

positive_test_input = """
I love this.
"""

In [13]:
def sentiment_analysis(input):
    user_sequences = tokenizer.texts_to_sequences([input])
    user_sequences_matrix = sequence.pad_sequences(user_sequences, maxlen=1225)
    prediction = model.predict(user_sequences_matrix)
    return prediction[0][0]

In [14]:
print("Probability that '{}' is positive: {}".format(negative_test_input, sentiment_analysis(negative_test_input)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step
Probability that '
this is bad junk.
' is positive: 0.010531649924814701


In [15]:
print("Probability that '{}' is positive: {}".format(positive_test_input, sentiment_analysis(positive_test_input)))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
Probability that '
I love this.
' is positive: 0.9518632292747498
